In [ ]:
import sys
import os
from datetime import datetime
from pyspark import SparkContext, SparkConf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

try:
    sc.stop()
except:
    pass

conf = SparkConf().setAppName("TrafficFeatureEngineering").setMaster("local[*]")
sc = SparkContext(conf=conf)

traffic_path = "traffic.csv"
raw_rdd = sc.textFile(traffic_path)

header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda line: line != header)

def parse_and_extract_features(line):
    try:
        parts = line.split(',')
        device_code = parts[0].strip()
        time_str = parts[4].strip()
        
        dt_obj = datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S")
        
        hour = dt_obj.hour
        dow = dt_obj.isoweekday()
        
        vector_idx = (dow - 1) * 24 + hour
        
        return ((device_code, vector_idx), 1)
    except:
        return None

features_rdd = data_rdd.map(parse_and_extract_features).filter(lambda x: x is not None)

slots_counts_rdd = features_rdd.reduceByKey(lambda a, b: a + b)

mapped_slots = slots_counts_rdd.map(lambda x: (x[0][0], (x[0][1], x[1])))

grouped_cameras = mapped_slots.groupByKey()

def create_168_vector(data_iterable):
    vector = [0] * 168
    for idx, count in data_iterable:
        if 0 <= idx < 168:
            vector[idx] = count
    return vector

final_vectors_rdd = grouped_cameras.mapValues(create_168_vector)
final_vectors_rdd.persist()

print(f"Total Cameras Processed: {final_vectors_rdd.count()}")
for device, vec in final_vectors_rdd.take(2):
    print(f"Device: {device}")
    non_zero_slots = [(i, v) for i, v in enumerate(vec) if v > 0]
    print(f"Non-zero slots: {non_zero_slots[:5]} ...")

In [ ]:
def l1_normalize(vector):
    total_sum = sum(vector)
    if total_sum == 0:
        return vector
    return [x / total_sum for x in vector]

normalized_rdd = final_vectors_rdd.mapValues(l1_normalize)
normalized_rdd.persist()

print(f"Total Normalized Vectors: {normalized_rdd.count()}")

sample = normalized_rdd.take(1)[0]
device_id = sample[0]
vec = sample[1]

print(f"Device: {device_id}")
print(f"Sum of vector elements: {sum(vec)}")
print(f"Vector preview: {vec[:10]}...")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyspark.mllib.linalg import Vectors
from pyspark.mllib.linalg.distributed import RowMatrix
from pyspark.mllib.stat import Statistics

vector_rdd = normalized_rdd.map(lambda x: Vectors.dense(x[1]))
vector_rdd.cache()

summary = Statistics.colStats(vector_rdd)
mean_vector = summary.mean()
bc_mean = sc.broadcast(mean_vector)

def center_vector(v):
    return Vectors.dense(v - bc_mean.value)

centered_rdd = vector_rdd.map(center_vector)
centered_rdd.cache()

mat = RowMatrix(centered_rdd)
svd = mat.computeSVD(168, computeU=False)

s = svd.s.toArray()
eigenvalues = s ** 2
total_variance = np.sum(eigenvalues)
explained_variance_ratio = eigenvalues / total_variance
cumulative_variance = np.cumsum(explained_variance_ratio)

k_pca = np.searchsorted(cumulative_variance, 0.90) + 1

print(f"Selected k_pca for 90% variance: {k_pca}")
print(f"Cumulative Variance at k={k_pca}: {cumulative_variance[k_pca-1]:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(range(1, 169), cumulative_variance, marker='.', linestyle='--')
plt.axhline(y=0.90, color='r', linestyle='-', label='90% Threshold')
plt.axvline(x=k_pca, color='g', linestyle='-', label=f'k_pca={k_pca}')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Analysis: Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.savefig('pca_variance_plot.png')
print("Plot saved as pca_variance_plot.png")

V = svd.V.toArray()
V_k = V[:, :k_pca]
bc_Vk = sc.broadcast(V_k)

def transform_to_pca(v):
    return np.dot(v, bc_Vk.value)

pca_transformed_rdd = centered_rdd.map(transform_to_pca)

print(f"\nFinal Transformed RDD Shape: {pca_transformed_rdd.count()} rows x {k_pca} columns")
sample_pca = pca_transformed_rdd.take(1)[0]
print(f"Sample PCA Vector (First 5 components): {sample_pca[:5]}")

In [ ]:
from pyspark.mllib.clustering import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt

# --- Sampling configuration: Use a random sample for evaluation ---
# For large input, use a 10000-record sample for this evaluation
# Full-data silhouette evaluation can be prohibitively expensive for large inputs
sample_count = 10000
total_count = pca_transformed_rdd.count()

if total_count > sample_count:
    fraction = sample_count / total_count
    # Take a random sample
    sampled_rdd = pca_transformed_rdd.sample(False, fraction, 42)
    X_local = np.array(sampled_rdd.collect())
else:
    X_local = np.array(pca_transformed_rdd.collect())

print(f"Data sampled for evaluation: {len(X_local)} records")

k_values = range(3, 13)
results = []
init_modes = [("KMeans (Random)", "random"), ("KMeans++", "k-means||")]

print(f"{'Method':<20} | {'K':<3} | {'Silhouette Score':<20}")
print("-" * 50)

best_k = 0
best_score = -1
best_method = ""

for method_name, mode in init_modes:
    scores = []
    for k in k_values:
        
        model = KMeans.train(pca_transformed_rdd, k, maxIterations=100, initializationMode=mode, seed=42)
        temp_rdd = sc.parallelize(X_local)
        predictions = model.predict(temp_rdd)
        labels = predictions.collect()
        
        
        if len(set(labels)) > 1: 
            score = silhouette_score(X_local, labels)
        else:
            score = -1
            
        scores.append(score)
        
        print(f"{method_name:<20} | {k:<3} | {score:.4f}")
        
        if score > best_score:
            best_score = score
            best_k = k
            best_method = method_name
            
    results.append((method_name, scores))

print("-" * 50)
print(f"Best Result: K={best_k} with Score={best_score:.4f} using {best_method}")

plt.figure(figsize=(10, 6))
for method_name, scores in results:
    plt.plot(k_values, scores, marker='o', label=method_name)

plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score (Sampled)')
plt.title('KMeans vs KMeans++ Silhouette Analysis')
plt.legend()
plt.grid(True)
plt.savefig('kmeans_silhouette_comparison.png')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def get_24h_profile(vector_168):
    matrix = np.array(vector_168).reshape(7, 24)
    return np.mean(matrix, axis=0)

combined_rdd = pca_transformed_rdd.zip(final_vectors_rdd.map(lambda x: x[1]))
labeled_rdd = combined_rdd.map(lambda x: (model.predict(x[0]), (x[0], x[1])))
labeled_rdd.persist()

cluster_counts = labeled_rdd.countByKey()
sorted_clusters = sorted(cluster_counts.items())

cluster_centers = model.clusterCenters

stats = []
for k in range(best_k):
    cluster_data = labeled_rdd.filter(lambda x: x[0] == k)
    
    avg_vec = cluster_data.map(lambda x: np.array(x[1][1])).reduce(lambda a, b: a + b)
    avg_vec = avg_vec / cluster_counts[k]
    
    center_vec = cluster_centers[k]
    
    def get_dist(record):
        pca_vec = record[1][0]
        dist = np.linalg.norm(pca_vec - center_vec)
        return (dist, record[1][1]) 
    
    rep_camera = cluster_data.map(get_dist).min(key=lambda x: x[0])
    
    stats.append({
        'cluster_id': k,
        'count': cluster_counts[k],
        'avg_profile': get_24h_profile(avg_vec),
        'rep_profile': get_24h_profile(rep_camera[1])
    })

plt.figure(figsize=(10, 6))
clusters = [s['cluster_id'] for s in stats]
counts = [s['count'] for s in stats]
plt.bar(clusters, counts, color='skyblue', edgecolor='black')
plt.title('1. Cluster Size Distribution')
plt.xlabel('Cluster ID')
plt.ylabel('Number of Cameras')
plt.xticks(clusters)
plt.savefig('plot_1_bar.png')
plt.show()

plt.figure(figsize=(12, 6))
for s in stats:
    plt.plot(s['avg_profile'], label=f"Cluster {s['cluster_id']}")
plt.title('2. Average 24-Hour Profile per Cluster')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Normalized Traffic')
plt.legend()
plt.grid(True)
plt.savefig('plot_2_profiles.png')
plt.show()

sample_data = labeled_rdd.sample(False, 0.1, 42).collect()
pca_x = [x[1][0][0] for x in sample_data] 
pca_y = [x[1][0][1] for x in sample_data] 
labels = [x[0] for x in sample_data]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(pca_x, pca_y, c=labels, cmap='tab10', alpha=0.6, s=10)
plt.title('3. PCA Scatter Plot (PC1 vs PC2)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Cluster ID')
plt.grid(True)
plt.savefig('plot_3_scatter.png')
plt.show()

num_clusters = len(stats)
cols = 2
rows = (num_clusters + 1) // 2
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for i, s in enumerate(stats):
    ax = axes[i]
    ax.plot(s['avg_profile'], 'b--', label='Cluster Avg', linewidth=2)
    ax.plot(s['rep_profile'], 'r-', label='Representative (Centroid)', linewidth=1.5)
    ax.set_title(f"Cluster {s['cluster_id']} (Size: {s['count']})")
    ax.set_xlabel("Hour")
    ax.legend()
    ax.grid(True)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plot_4_representatives.png')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from operator import add
from pyspark.mllib.clustering import KMeans


def parse_for_stats(line):
    try:
        parts = line.split(',')
        # according to the supplied data: 0=Device, 4=Time, 6=Plate
        return (parts[0].strip(), parts[4].strip(), parts[6].strip())
    except:
        return None

parsed_rdd = data_rdd.map(parse_for_stats).filter(lambda x: x is not None)
parsed_rdd.cache()
print(f"parsed_rdd created with {parsed_rdd.count()} records.")


total_passes = parsed_rdd.map(lambda x: (x[0], 1)).reduceByKey(add)

unique_plates = parsed_rdd.map(lambda x: (x[0], x[2])) \
                          .distinct() \
                          .map(lambda x: (x[0], 1)) \
                          .reduceByKey(add)

def get_hour_key(record):
    try:
        h = int(record[1].split(' ')[1].split(':')[0])
        return ((record[0], h), 1)
    except:
        return None

hourly_counts = parsed_rdd.map(get_hour_key).filter(lambda x: x is not None).reduceByKey(add)

peak_hours = hourly_counts.map(lambda x: (x[0][0], (x[0][1], x[1]))) \
                          .reduceByKey(lambda a, b: a if a[1] > b[1] else b) \
                          .mapValues(lambda x: x[0])

stats_joined = total_passes.join(unique_plates).join(peak_hours)
camera_stats_rdd = stats_joined.mapValues(lambda x: (x[0][0], x[0][1], x[1]))

def generate_outputs(model_instance, k):
    devices = final_vectors_rdd.map(lambda x: x[0])
    device_pca = devices.zip(pca_transformed_rdd)
    
    cluster_preds = device_pca.map(lambda x: (x[0], model_instance.predict(x[1])))
    
    cam_cluster_data = cluster_preds.join(camera_stats_rdd) \
                                    .map(lambda x: (x[0], x[1][0], x[1][1][0], x[1][1][1], x[1][1][2])) \
                                    .collect()
    
    df_clusters = pd.DataFrame(cam_cluster_data, columns=['DeviceCode', 'ClusterID', 'TotalPasses', 'UniquePlates', 'PeakHour'])
    
    def to_24h_norm(vec_168):
        arr = np.array(vec_168).reshape(7, 24)
        sum_24 = np.sum(arr, axis=0)
        total = np.sum(sum_24)
        if total > 0:
            return sum_24 / total
        return sum_24

    dev_vec_cluster = final_vectors_rdd.join(cluster_preds)
    
    cluster_profiles = dev_vec_cluster.map(lambda x: (x[1][1], (to_24h_norm(x[1][0]), 1))) \
                                      .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                                      .mapValues(lambda x: x[0] / x[1]) \
                                      .collect()
    
    profile_rows = []
    for cid, profile in cluster_profiles:
        for h in range(24):
            profile_rows.append({'ClusterID': cid, 'Hour': h, 'AvgPHour': profile[h]})
            
    df_profiles = pd.DataFrame(profile_rows)
    
    return df_clusters, df_profiles

try:
    k_target = best_k
except NameError:
    k_target = 5


model_km = KMeans.train(pca_transformed_rdd, k_target, maxIterations=100, initializationMode="random", seed=42)
df_km_cam, df_km_prof = generate_outputs(model_km, k_target)

model_kmpp = KMeans.train(pca_transformed_rdd, k_target, maxIterations=100, initializationMode="k-means||", seed=42)
df_kmpp_cam, df_kmpp_prof = generate_outputs(model_kmpp, k_target)

df_km_cam.to_csv("kmeans_camera_clusters.csv", index=False)
df_km_prof.to_csv("kmeans_cluster_hour_profile.csv", index=False)

with pd.ExcelWriter('Analysis_Results.xlsx') as writer:
    df_km_cam.to_excel(writer, sheet_name='KM_CameraClusters', index=False)
    df_km_prof.to_excel(writer, sheet_name='KM_HourProfile', index=False)
    df_kmpp_cam.to_excel(writer, sheet_name='KMpp_CameraClusters', index=False)
    df_kmpp_prof.to_excel(writer, sheet_name='KMpp_HourProfile', index=False)

print("Files generated successfully.")